# Logistic Regression

This notebook evaluates Logistic Regression as the first supervised machine learning model for HDFS anomaly detection.

Unlike the statistical baseline, Logistic Regression learns a set of coefficients from the labeled training data rather than relying on manually defined decision rules. The model estimates the probability that each HDFS block belongs to either the normal or anomalous class based on the engineered feature values.

Each of the four engineered feature sets is evaluated independently using the same training and testing splits created during the model preparation stage. Model performance is assessed using Accuracy, Precision, Recall, and F1 Score, allowing direct comparison with the statistical rule based baseline.

## Environment Setup

The required libraries are imported, and the project directory structure is initialized. The prepared training and testing datasets generated during model preparation will be loaded from the `data/processed/splits/` directory.

In [1]:
# Import required libraries

from pathlib import Path

import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)

In [2]:
# Determine the project root directory

current_path = Path.cwd()

if current_path.name == "notebooks":
    project_root = current_path.parent
else:
    project_root = current_path

processed_dir = project_root / "data" / "processed"
splits_dir = processed_dir / "splits"

results_dir = project_root / "results"
logistic_results_dir = results_dir / "logistic_regression"

logistic_results_dir.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Splits folder:", splits_dir)
print("Results folder:", logistic_results_dir)

Project root: c:\Users\taman\Documents\hdfs-anomaly-detection-study
Splits folder: c:\Users\taman\Documents\hdfs-anomaly-detection-study\data\processed\splits
Results folder: c:\Users\taman\Documents\hdfs-anomaly-detection-study\results\logistic_regression


## Load Prepared Training and Testing Data

The prepared train and test splits are loaded for each engineered feature set. These datasets were created during model preparation and will allow Logistic Regression to be evaluated using the same data splits as the statistical baseline.

In [3]:
# Load prepared train-test splits for each feature set

logistic_data = {}

for i in range(1, 5):

    feature_name = f"feature_set_{i}"
    feature_dir = splits_dir / feature_name

    logistic_data[feature_name] = {
        "X_train": pd.read_csv(feature_dir / "X_train.csv"),
        "X_test": pd.read_csv(feature_dir / "X_test.csv"),
        "y_train": pd.read_csv(feature_dir / "y_train.csv").squeeze(),
        "y_test": pd.read_csv(feature_dir / "y_test.csv").squeeze(),
    }

In [4]:
# Verify loaded dataset dimensions

for name, data in logistic_data.items():

    print(f"\n{name}")
    print(f"X_train: {data['X_train'].shape}")
    print(f"X_test : {data['X_test'].shape}")
    print(f"y_train: {data['y_train'].shape}")
    print(f"y_test : {data['y_test'].shape}")


feature_set_1
X_train: (460048, 29)
X_test : (115013, 29)
y_train: (460048,)
y_test : (115013,)

feature_set_2
X_train: (460048, 31)
X_test : (115013, 31)
y_train: (460048,)
y_test : (115013,)

feature_set_3
X_train: (460048, 311)
X_test : (115013, 311)
y_train: (460048,)
y_test : (115013,)

feature_set_4
X_train: (460048, 1082)
X_test : (115013, 1082)
y_train: (460048,)
y_test : (115013,)


## Train Logistic Regression Models

A separate Logistic Regression model is trained for each engineered feature set using the prepared training data.

Because the HDFS dataset is highly imbalanced, with substantially more normal than anomalous blocks, balanced class weights are used during training. This allows the model to place greater emphasis on correctly identifying the minority anomalous class while reducing bias toward predicting the majority normal class.

In [ ]:
# Train a Logistic Regression model for each engineered feature set

# Create a dictionary to store the trained models
logistic_models = {}

# Train one model for each feature set
for name, data in logistic_data.items():

    # Retrieve the training features and labels
    X_train = data["X_train"]
    y_train = data["y_train"]

    # Create the Logistic Regression model
    # Use balanced class weights to account for the imbalanced dataset. 
    # Increase the maximum iterations to ensure convergence.
    # Set a random state for reproducible results.
    model = LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42
    )

    # Learn the relationship between the engineered features and labels
    model.fit(X_train, y_train)

    # Store the trained model
    logistic_models[name] = model

In [6]:
# Confirm that all Logistic Regression models were successfully trained

for name in logistic_models:

    print(f"{name}: Model trained successfully")

feature_set_1: Model trained successfully
feature_set_2: Model trained successfully
feature_set_3: Model trained successfully
feature_set_4: Model trained successfully


## Evaluate Logistic Regression Performance

Each trained Logistic Regression model is evaluated using the unseen testing data for its corresponding feature set.

Predicted class labels are compared with the ground truth testing labels to calculate Accuracy, Precision, Recall, and F1 Score. These evaluation metrics will be used to compare the performance of Logistic Regression across all engineered feature sets and against the statistical rule based baseline.

In [7]:
# Evaluate each trained Logistic Regression model on the testing data

# Create a list to store the evaluation results
logistic_results = []

# Evaluate one model for each feature set
for name, model in logistic_models.items():

    # Retrieve the testing features and labels
    X_test = logistic_data[name]["X_test"]
    y_test = logistic_data[name]["y_test"]

    # Predict the class label for each testing block
    y_pred = model.predict(X_test)

    # Calculate evaluation metrics
    logistic_results.append({
        "Feature_Set": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "F1": f1_score(y_test, y_pred, zero_division=0)
    })

# Convert the results into a DataFrame
logistic_results = pd.DataFrame(logistic_results)

# Display the evaluation results
display(logistic_results)

,Feature_Set,Accuracy,Precision,Recall,F1
0,feature_set_1,0.998870,0.963366,0.999406,0.981055
1,feature_set_2,0.998861,0.963621,0.998812,0.980901
2,feature_set_3,0.999261,0.976764,0.998515,0.987520
3,feature_set_4,0.999357,0.979616,0.998812,0.989121


In [8]:
# Export the Logistic Regression results

logistic_results_path = (
    logistic_results_dir / "logistic_regression_results.csv"
)

logistic_results.to_csv(
    logistic_results_path,
    index=False
)

print("Logistic Regression results saved to:")
print(logistic_results_path)

Logistic Regression results saved to:
c:\Users\taman\Documents\hdfs-anomaly-detection-study\results\logistic_regression\logistic_regression_results.csv


## Logistic Regression Summary

Logistic Regression was successfully trained and evaluated across all four engineered feature sets. Compared to the statistical rule based baseline, Logistic Regression achieved a substantial improvement in anomaly detection performance. The highest statistical baseline F1 score was **0.4125**, whereas Logistic Regression achieved a best F1 score of **0.9891** using Feature Set 4, representing an improvement of approximately **140%**.

Unlike the statistical baseline, which experienced decreasing performance as additional engineered features were introduced, Logistic Regression demonstrated the opposite trend. The F1 score increased from **0.9811** with Feature Set 1 to **0.9891** with Feature Set 4. Although the numerical improvement appears modest, it was accompanied by consistently high Precision and Recall, indicating that the additional sequential features provided meaningful predictive information without sacrificing classification performance.

Feature Set 4 produced the strongest overall results, achieving **99.94% Accuracy**, **97.96% Precision**, **99.88% Recall**, and an **F1 Score of 0.9891**. These results suggest that the three-event sequence features capture additional behavioral patterns that are effectively utilized by a supervised learning model.

The contrast between the statistical baseline and Logistic Regression highlights an important finding of this study. Richer sequential feature representations did not improve a simple threshold based decision rule, but they significantly improved the performance of a supervised classification model capable of learning relationships among the engineered features. This observation provides strong evidence that the value of sequential feature engineering depends not only on the features themselves, but also on the learning capability of the underlying model.